# Week 1 Notebook: One Question, One Test, One Interpretation


## Introduction

In practice, we usually observe a **sample**, not the full **population**.
Because of that, our conclusions always include **uncertainty**.

Statistical inference helps us:

- estimate population quantities from sample data,
- and test simple questions in a careful way.

This notebook shows one clean inferential workflow for Week 1.


## Our One Question

**Question:** Do `setosa` and `versicolor` have different average sepal lengths?

Why this question is suitable for Week 1:

- two groups,
- one numeric variable,
- one clear inferential test (independent two-sample t-test).


## Step 1 - Load Data and Basic Setup

### What we are doing

- importing required libraries,
- loading the Iris dataset,
- preparing readable group labels.

### Why this matters

A clean setup makes the full workflow easy to follow and reproduce.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from scipy import stats

# Reproducibility
np.random.seed(42)
sns.set_theme(style="whitegrid")

# Load dataset
iris = load_iris(as_frame=True)
df = iris.frame.copy()

# Map target numbers to species names
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
df = df.drop(columns=["target"])


### Interpretation

The dataset is now in a readable form with one categorical label (`species`) and numeric features.


## Step 2 - EDA: Inspect, Summarize, Visualize

### What we are trying to learn

Before testing, we first understand the data structure and basic patterns.

### What should we expect

We should see two groups (`setosa`, `versicolor`) with some overlap but different centers.


In [ ]:
# Keep only the two groups we want to compare
two_groups = df[df["species"].isin(["setosa", "versicolor"])].copy()

# Quick inspection
two_groups.head()


In [ ]:
# Identify variable types
print("Variable types:")
print(two_groups.dtypes)

# Descriptive stats for the main variable
summary = two_groups.groupby("species")["sepal length (cm)"].agg(["count", "mean", "std", "min", "max"])
summary


In [ ]:
# Visualize sepal length by group
plt.figure(figsize=(8, 4))
sns.boxplot(data=two_groups, x="species", y="sepal length (cm)", palette="Set2")
plt.title("Sepal Length by Species (Setosa vs Versicolor)")
plt.xlabel("Species")
plt.ylabel("Sepal length (cm)")
plt.show()


### Interpretation

- `species` is categorical; `sepal length (cm)` is quantitative.
- Group means and boxplots suggest a difference, but visual evidence alone is not enough.
- We still need formal inference.

### Common mistake

Concluding significance from a plot without a statistical test.


## Step 3 - Hypotheses (Simple Wording)

- **H0 (null hypothesis):** The average sepal length is the same for `setosa` and `versicolor`.
- **H1 (alternative hypothesis):** The average sepal length is different for `setosa` and `versicolor`.


## Step 4 - Confidence Interval (Estimate + Uncertainty)

### What we are doing

We compute a 95% confidence interval for the difference in means:

\[
	ext{mean(setosa)} - 	ext{mean(versicolor)}
\]

### Why this matters

A confidence interval gives both:

- an estimated difference,
- and uncertainty around that estimate.


In [ ]:
# Extract the two samples
setosa = two_groups[two_groups["species"] == "setosa"]["sepal length (cm)"]
versicolor = two_groups[two_groups["species"] == "versicolor"]["sepal length (cm)"]

# Estimate difference in means
mean_diff = setosa.mean() - versicolor.mean()

# Standard error for Welch-style two-sample mean difference
se_diff = np.sqrt(setosa.var(ddof=1)/len(setosa) + versicolor.var(ddof=1)/len(versicolor))

# Welch-Satterthwaite degrees of freedom
df_welch = (setosa.var(ddof=1)/len(setosa) + versicolor.var(ddof=1)/len(versicolor))**2 / (
    ((setosa.var(ddof=1)/len(setosa))**2)/(len(setosa)-1) +
    ((versicolor.var(ddof=1)/len(versicolor))**2)/(len(versicolor)-1)
)

# 95% CI critical value and interval
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha/2, df=df_welch)
ci_low = mean_diff - t_crit * se_diff
ci_high = mean_diff + t_crit * se_diff

print(f"Estimated mean difference (setosa - versicolor): {mean_diff:.3f} cm")
print(f"95% CI: [{ci_low:.3f}, {ci_high:.3f}] cm")


### Interpretation

- The interval shows a plausible range for the population mean difference.
- If 0 is outside the interval, that suggests evidence of a true difference.

### Important note

A 95% CI is about long-run procedure reliability, not a guaranteed probability about this one fixed interval.


## Step 5 - One Statistical Test Only (Welch Two-Sample t-test)

### What we are doing

We run exactly one inferential test: a Welch independent t-test.

### Why this test

- two independent groups,
- numeric outcome,
- does not require equal variance assumption.


In [ ]:
t_stat, p_value = stats.ttest_ind(setosa, versicolor, equal_var=False)

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.8f}")


### Interpretation

The t-statistic measures standardized separation between group means.
The p-value tells us how unusual this result would be if H0 were true.


## Step 6 - How to Interpret the p-value Correctly

- The p-value is the probability of observing data this extreme (or more extreme) **if H0 is true**.
- A small p-value gives evidence **against H0**.
- It does **not** prove H1.
- It does **not** measure practical importance.
- It should be combined with effect size/context (and CI) before making claims.


## Final Plain-Language Interpretation

From this sample, there is strong evidence that average sepal length differs between `setosa` and `versicolor`.

We base this on:

- a small p-value from one appropriate t-test,
- and a confidence interval for the mean difference.

We still acknowledge uncertainty because we are reasoning from a sample.
We also avoid causal claims: this is a group comparison, not a causal experiment.


## Quick Concept Check (with answers)

**Q1:** Why do we need inference instead of only sample means?  
**A1:** Because sample means vary from sample to sample; inference quantifies uncertainty.

**Q2:** If p < 0.05, does that prove H1 is true?  
**A2:** No. It means the observed result is unlikely under H0.

**Q3:** Why include a confidence interval?  
**A3:** It shows both estimated effect and uncertainty range.


## Optional Short Practice

Repeat the same workflow for `petal width (cm)` between `versicolor` and `virginica`:

1. EDA,
2. hypotheses,
3. confidence interval,
4. one Welch t-test,
5. plain-language interpretation.
